## PROCESSAMENTO DOS DADOS
Este notebook visa ler os dados de bares e restaurantes, extrair suas coordenadas geográficas usando a API da ```geocode.maps.co``` e formar um novo conjunto de dados apenas com as informações necessárias.

### 1- Leitura e Filtragem inicial
Aqui usamos a biblioteca pandas para ler o arquivo ```bares.csv``` e filtrar algumas colunas que não vamos precisar, além de mostrar quais os tipos de descrição de logradouro, pois é necessário convertê-los para o nome completo na hora da query.

In [3]:
import pandas as pd
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PRC_DIR = ROOT / "data" / "processed"

In [3]:
bar_table = pd.read_csv(f"{RAW_DIR}/bares.csv")
colunas_rem = ["CNAE_PRINCIPAL", "DESCRICAO_CNAE_PRINCIPAL"]
bar_table = bar_table.drop(columns=colunas_rem)
bar_table.head(5)

,_id,DESC_LOGRADOURO,NOME_LOGRADOURO,NUMERO_IMOVEL,COMPLEMENTO,NOME_BAIRRO,NOME,NOME_FANTASIA
0,12,AVE,PORTUGAL,4316,NaN,JARDIM ATLANTICO,BAR E CAFE DONATELO LTDA,NaN
1,32,RUA,VINTE E OITO DE SETEMBRO,606,NaN,ESPLANADA,PALLATE COMERCIO DE PRODUTOS ALIMENTICIOS LTDA,PALLATE
2,40,AVE,EDGAR DA MATA MACHADO,390,A,DOM CABRAL,FABIO KALKS KOSCKY,BAR DO FABIO
3,58,AVE,DOS BANDEIRANTES,1839,NaN,CRUZEIRO,NECTAR DA SERRA LTDA,NECTAR DA SERRA
4,60,RUA,DO OURO,1428,NaN,SERRA,MERCEARIA DA SERRA LTDA,BAR E MERCEARIA DA SERRA


In [4]:
# Análise de descrição de logradouros
bar_table[["_id", "DESC_LOGRADOURO"]].groupby("DESC_LOGRADOURO").count()

,_id
DESC_LOGRADOURO,
ALA,32
AVE,4604
BEC,114
EST,25
PCA,190
ROD,171
RUA,15709
TRV,10
VDP,1


In [5]:
null_report = (bar_table.isna().mean().mul(100).round(2).sort_values(ascending=False).reset_index())
null_report.columns = ["column", "null_percent"]
null_report[null_report["null_percent"] > 0].head(20)

,column,null_percent
0,COMPLEMENTO,52.98
1,NOME_FANTASIA,45.29


### 2- Extração das Coordenadas
A extração das coordenadas a partir dos endereços é feita pela API da ```geocode.maps.co```. É necessário ter uma conta e ajustar os valores dentro das células de acordo com a sua conta.
No entanto, não recomendo rodar o script se ```cords.json``` já existir em ```data/processed```, pois leva horas para extrair cada coordenada geográfica de todos os endereços.

In [12]:
import time
import json
import os
import requests as req

In [7]:
CORDS_DIR = f"{PRC_DIR}/cords.json"
FDATA_DIR = f"{PRC_DIR}/fulldata.json"

# Parâmetros - Altere os valores aqui
api_key = ""                # Coloque a sua API key aqui
sleep_time = 0.25           # Tempo entre uma request e outra, varia de acordo com a licença
CONSULTAS = len(bar_table)  # len(bar_table) para tudo

In [ ]:
url = "https://geocode.maps.co/search"
cols = ["DESC_LOGRADOURO", "NOME_LOGRADOURO", "NUMERO_IMOVEL"]

# Dicionários que serão escritos nos json mais tarde
if os.path.exists(CORDS_DIR):
    try:
        with open(CORDS_DIR, "r", encoding="utf-8") as f:
            cords_data = json.load(f)
    except Exception as e:
        print("Unable to load previous data:", e)
        cords_data = {}

else:
    cords_data = {}

# Mantém contagem de sucessos e falhas nas consultas
successes = 0
fails = 0

# Consultas
for i in range(CONSULTAS):
    # Caso o endereço já tenha sido processado
    if str(i) in cords_data:
        continue

    bar = bar_table.iloc[i]
    bar_name = bar["NOME"]
    bar_fantasia = bar["NOME_FANTASIA"]
    bar_fantasia = str(bar_fantasia) if pd.notna(bar_fantasia) else ""

    desc, logr, numi = [bar[cols[j]] for j in range(len(cols))]

    # Conversão do nome inteiro do tipo de logradouro
    match desc:
        case "AVE":
            desc = "AVENIDA"
        case "ALA":
            desc = "ALAMEDA"
        case "BEC":
            desc = "BECO"
        case "EST":
            desc = "ESTRADA"
        case "PCA":
            desc = "PRAÇA"
        case "ROD":
            desc = "RODOVIA"
            logr = "BR356"
        case "RUA":
            desc = "RUA"
        case "TRV":
            desc = "TRAVESSA"
        case "VDP":
            desc = ""
        case "VIA":
            desc = "VIADUTO"

    address = f"{desc} {logr}, {numi}, Belo Horizonte, Minas Gerais, Brazil"
    params = {"q": address, "api_key": api_key}

    success = False

    for t in range(2):
        try:
            query = req.get(url, params=params, timeout=10)
            query.raise_for_status()
            data = query.json()

            # cords_full_data[str(i)] = {"NOME": bar_name, "NOME_FANTASIA": bar_fantasia, "ENDERECO": address, "GEOCODE": data}
            if data:
                lat = data[0].get("lat")
                lon = data[0].get("lon")
                cords_data[str(i)] = {"NOME": bar_name, "NOME_FANTASIA": bar_fantasia, "ENDERECO": address, "LATITUDE": lat, "LONGITUDE": lon}

            if i%200 == 0:  # Evita escrita em disco todo loop
                with open(CORDS_DIR, "w", encoding="utf-8") as f:
                    json.dump(cords_data, f, ensure_ascii=False, indent=4)

            success = True
            successes += 1
            print(f"[{i}] {bar_name}: query successful")

            break

        except Exception as e:
            print(f"Failed line {i} - {bar_name}: {e}")
            time.sleep(sleep_time)

    # Falha crítica - nenhuma das tentativas sucedeu
    if success == False:
        print(f"Critical failure on line {i}")
        fails += 1

    time.sleep(sleep_time)    # Limite de consultas da API

# Dump final
with open(CORDS_DIR, "w", encoding="utf-8") as f:
    json.dump(cords_data, f, ensure_ascii=False, indent=4)

# with open(FDATA_DIR, "w", encoding="utf-8") as f:
    # json.dump(cords_full_data, f, ensure_ascii=False, indent=4)

print("Sucessos:", successes)
print("Falhas:", fails)

[17200] 60.792.336 DANIEL PEREIRA DE SOUZA: query successful
[17210] 60.898.339 FLAVIO SALTARA: query successful
[17214] 60.832.757 KARINA APARECIDA DA SILVA: query successful
[17275] 61.663.666 CAIO HENRIQUE LOPES FERREIRA: query successful
[17283] 61.801.634 RAYANE REZENDE SANTOS DE OLIVEIRA: query successful
[17289] ALTEROSAS BAR E RESTAURANTE LTDA: query successful
[17290] CURRAL DA SERRA RESTAURANTE LTDA COUNTRY CLUB: query successful
[17294] 61.653.975 DENNER CESAR FERREIRA GOMES: query successful
[17296] CLA ESPACO E CULTURA LTDA: query successful
[17394] 61.475.910 ALUISIO MACHADO DE OLIVEIRA: query successful
[17406] COPO MINEIRO LTDA: query successful
[17409] 61.598.501 NATHALIA CRISTINA PEREIRA VERISSIMO COSTA: query successful
[17437] 61.309.653 SILBERT RIBEIRO ALVES: query successful
[17503] SO JOAO BUTIQUIM LTDA: query successful
[17513] 55.119.620 IARA PERPETUO GOMES: query successful
[17527] ANDU RESTAURANTE LTDA: query successful
[17560] 61.130.451 DAIANA GONCALVES DE 

### 3- Construção de uma Nova Base de Dados
Com base nas coordenadas extraídas, agora construímos uma nova tabela com os endereços e coordenadas para ser usada na simulação.

O índice da tabela é renomeado para _old_index_ porque o índice não representa o número da linha nesta nova tabela, mas sim o número da linha correspondente na tabela antiga, para que seja possível cruzar os dados depois caso necessário.

In [15]:
# Carregamento dos dados
if os.path.exists(CORDS_DIR):
    try:
        with open(CORDS_DIR, "r", encoding="utf-8") as f:
            cords_data = json.load(f)
    except Exception as e:
        print("Unable to load previous data:", e)

new_bar_table = pd.DataFrame.from_dict(cords_data, orient='index')

# Ajustes
new_bar_table.index.name = "old_index"
new_bar_table["LATITUDE"] = new_bar_table["LATITUDE"].astype(float)
new_bar_table["LONGITUDE"] = new_bar_table["LONGITUDE"].astype(float)

new_bar_table

,NOME,NOME_FANTASIA,ENDERECO,LATITUDE,LONGITUDE
old_index,,,,,
0,BAR E CAFE DONATELO LTDA,,"AVENIDA PORTUGAL, 4316, Belo Horizonte, Minas ...",-19.843947,-43.958138
1,PALLATE COMERCIO DE PRODUTOS ALIMENTICIOS LTDA,PALLATE,"RUA VINTE E OITO DE SETEMBRO, 606, Belo Horizo...",-19.911048,-43.907020
2,FABIO KALKS KOSCKY,BAR DO FABIO,"AVENIDA EDGAR DA MATA MACHADO, 390, Belo Horiz...",-19.920171,-43.997347
3,NECTAR DA SERRA LTDA,NECTAR DA SERRA,"AVENIDA DOS BANDEIRANTES, 1839, Belo Horizonte...",-19.949169,-43.920703
4,MERCEARIA DA SERRA LTDA,BAR E MERCEARIA DA SERRA,"RUA DO OURO, 1428, Belo Horizonte, Minas Gerai...",-19.947210,-43.918542
...,...,...,...,...,...
20854,68.809.777 IRISNEIDE ALVES PEREIRA,,"RUA EMILIA MARZANO MATIAS, 115, Belo Horizonte...",-19.803473,-43.985905
20855,68.819.988 KAREN ALEXANDRA PEREIRA DA CRUZ LIMA,,"RUA ACIDENTAL, 46, Belo Horizonte, Minas Gerai...",-19.934670,-43.907813
20856,68.828.610 LUCINEIA APARECIDA FERREIRA ALVES,,"RUA BELA VISTA, 251, Belo Horizonte, Minas Ger...",-19.919168,-43.982568


In [16]:
# Número e porcentagem de perdas de endereços durante o processo de aquisição das coordenadas
bar_og = len(bar_table)
bar_cd = len(new_bar_table)
print("Endereços originais:", bar_og)
print("Endereços extraídos:", bar_cd)
print("Perda:", bar_og - bar_cd)
print("Perda em %:", round(bar_cd/bar_og, 3))

Endereços originais: 20859
Endereços extraídos: 19867
Perda: 992
Perda em %: 0.952


In [18]:
# Salvamento do csv novo
new_bar_table.to_csv(PRC_DIR / 'new_bares.csv', index=True)
test = pd.read_csv(PRC_DIR / 'new_bares.csv')
test.tail(5)

,old_index,NOME,NOME_FANTASIA,ENDERECO,LATITUDE,LONGITUDE
19862,20854,68.809.777 IRISNEIDE ALVES PEREIRA,NaN,"RUA EMILIA MARZANO MATIAS, 115, Belo Horizonte...",-19.803473,-43.985905
19863,20855,68.819.988 KAREN ALEXANDRA PEREIRA DA CRUZ LIMA,NaN,"RUA ACIDENTAL, 46, Belo Horizonte, Minas Gerai...",-19.934670,-43.907813
19864,20856,68.828.610 LUCINEIA APARECIDA FERREIRA ALVES,NaN,"RUA BELA VISTA, 251, Belo Horizonte, Minas Ger...",-19.919168,-43.982568
19865,20857,68.828.670 ANTONIO MENEZES DA SILVA,NaN,"RUA ERICO VERISSIMO, 2373, Belo Horizonte, Min...",-19.815254,-43.969663
19866,20858,SO COMBOS 2 COMERCIO LTDA,SO COMBOS,"RUA BAVIERA, 119, Belo Horizonte, Minas Gerais...",-19.887985,-43.948916
